# 02 - Silver: England Premier League - join all seasons into one table

First league of the per-league Silver approach (see docs/schema_notes.md
for why: the old per-season Bronze tables hit the Unity Catalog quota).
Reads every season CSV for this one league straight from the Bronze file
copy, unions them into a single table, and does a quick EDA pass to see
how much column drift actually exists within just this league's 21
seasons (should be far less than the 46 layouts across ALL leagues).

To do the next league: copy this notebook, change COUNTRY/DIVISION below,
change the output table name to match.

Writes to: `workspace.silver.england_premier_league`

In [ ]:
import csv
import io
import os
import pandas as pd

BRONZE_PATH = "/Volumes/workspace/default/football_bronze"
COUNTRY = "england"
DIVISION = "Premier League"
OUTPUT_TABLE = "workspace.silver.england_premier_league"

LEAGUE_DIR = f"{BRONZE_PATH}/main_leagues/{COUNTRY}/{DIVISION}"

## Step 1: robust CSV reader (same encoding + ragged-row handling as Bronze)

In [ ]:
def read_csv_robust(path):
    raw = None
    for enc in ("utf-8-sig", "cp1252"):
        try:
            with open(path, encoding=enc, newline="") as fh:
                raw = fh.read()
            break
        except UnicodeDecodeError:
            continue

    rows = list(csv.reader(io.StringIO(raw)))
    header = rows[0]
    ncols = len(header)
    fixed_rows = [r[:ncols] + [""] * (ncols - len(r)) for r in rows[1:]]
    df = pd.DataFrame(fixed_rows, columns=header)

    ghost = [c for c in df.columns if not c or str(c).startswith("Unnamed:")]
    df = df.drop(columns=ghost)

    # csv.reader gives "" for missing cells - treat as real nulls, not the
    # literal string "", so unioning seasons with different columns doesn't
    # produce a mix of "" and NaN for the same kind of missing value
    df = df.replace("", None)

    # Some files have a trailing blank line, which becomes an all-null row
    return df.dropna(how="all")

## Step 2: read every season for this league, tag with source_season, union

In [ ]:
season_files = sorted(f for f in os.listdir(LEAGUE_DIR) if f.endswith(".csv"))

season_dfs = []
for fname in season_files:
    season = fname.replace(".csv", "")
    df = read_csv_robust(os.path.join(LEAGUE_DIR, fname))
    df["source_season"] = season
    season_dfs.append(df)
    print(f"  {season}: {df.shape[0]} rows, {df.shape[1]} columns")

# sort=False keeps column order stable; concat aligns by column name and
# fills missing columns with null for seasons that don't have them
combined = pd.concat(season_dfs, ignore_index=True, sort=False)
print(f"\nCombined: {combined.shape[0]} rows, {combined.shape[1]} columns across {len(season_files)} seasons")

## Step 3: EDA - how much column drift is there within this one league?

In [ ]:
# Which columns aren't present in every season (drift within just this league)
per_season_cols = {df["source_season"].iloc[0]: set(df.columns) for df in season_dfs}
all_cols = set().union(*per_season_cols.values())
core_cols = set.intersection(*per_season_cols.values())

print(f"Columns present in every season (core): {len(core_cols)}")
print(f"Columns present in at least one season: {len(all_cols)}")
print(f"Columns that vary by season: {len(all_cols - core_cols)}\n")

for col in sorted(all_cols - core_cols):
    present_in = [s for s, cols in per_season_cols.items() if col in cols]
    print(f"  {col}: present in {len(present_in)}/{len(season_files)} seasons")

In [ ]:
# Sanity checks before writing
print("Any fully-duplicate rows:", combined.duplicated().sum())
print("Null count in core match fields:")
print(combined[[c for c in ["HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"] if c in combined.columns]].isnull().sum())

## Step 4: write the combined table

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

combined = combined.astype(object).where(combined.notna(), None)
spark_df = spark.createDataFrame(combined)
spark_df.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)

print(f"Wrote {OUTPUT_TABLE}: {combined.shape[0]} rows, {combined.shape[1]} columns")